                # GEPA

                Use textual failure feedback to evolve instructions, preserving reflection logs and best outputs along the way.

                **Use it when:** Your metric can explain errors, not merely score them, and you want a detailed instruction that encodes those lessons.

                **What compilation changes:** In this rerun GEPA learned a long standalone instruction and no final demonstrations.

                Important configuration in this benchmark:

                - six full evaluations in the full profile
- reflection minibatches of three
- feedback metric returns both score and diagnosis
- seed 42; merge disabled for a bounded run

                The 74-row dataset and pair-grouped train/validation/test membership are frozen.
                The test partition is deliberately baseline-adversarial, so these scores teach
                optimizer tradeoffs; they are not a general-purpose AI-detector leaderboard.

In [1]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "chapter06").is_dir() else cwd.parent
if not (REPO_ROOT / "chapter06" / "results" / "benchmark_summary.json").exists():
    raise RuntimeError("Run this notebook from the repository or chapter06 directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from chapter06.notebook_support import (
    artifact_paths,
    benchmark_snapshot,
    learned_program_preview,
    verify_prompt_artifact,
)

OPTIMIZER = 'gepa'
RUN_MODE = os.getenv("CHAPTER06_RUN", "inspect").lower()
print(f"mode={RUN_MODE!r}; optimizer={OPTIMIZER!r}")
print("safe default: inspect checked-in full-run artifacts; no API calls")

mode='inspect'; optimizer='gepa'
safe default: inspect checked-in full-run artifacts; no API calls


                ## Compile shape

                This is the essential DSPy call used by the shared runner (setup variables omitted):

                ```python
                optimizer = dspy.GEPA(
    metric=feedback_metric, max_full_evals=profile.gepa_max_full_evals,
    reflection_minibatch_size=3, reflection_lm=reflection_lm,
    use_merge=False, track_best_outputs=True, seed=42,
)
optimized_detector = optimizer.compile(detector, trainset=trainset, valset=valset)
                ```

                `compile` returns a program. Calling that program on the untouched test examples is
                a separate phase; the notebook reports optimization cost/time separately from inference latency.

In [2]:
print(benchmark_snapshot(OPTIMIZER))
print()
print(artifact_paths(OPTIMIZER))

GEPA — frozen full-profile rerun
status: completed
task model: openai/gpt-5.6-luna
test accuracy: 90.0%
uplift: +40.0 points vs Luna reference
optimization: $1.5658 and 15.0min
inference latency: mean 1.78s; p95 2.32s
reload checks: prompt=True; model=None; predictions=3/3 labels
complete run: chapter06/results/runs/full/gepa/20260715T024939.319893Z

Complete artifacts:
- complete_output: chapter06/results/runs/full/gepa/20260715T024939.319893Z/complete_output.txt
- lm_history: chapter06/results/runs/full/gepa/20260715T024939.319893Z/lm_history.jsonl
- metrics: chapter06/results/runs/full/gepa/20260715T024939.319893Z/metrics.json
- predictions: chapter06/results/runs/full/gepa/20260715T024939.319893Z/predictions.jsonl
- program: chapter06/results/runs/full/gepa/20260715T024939.319893Z/program.json
- prompts: chapter06/results/runs/full/gepa/20260715T024939.319893Z/prompts.json
- canonical program: chapter06/optimized_programs/final/gepa.json
- canonical prompt: chapter06/results/final_

## Read the result

The preview is deliberately truncated; follow the prompt path for the complete learned rule set and `optimizer_logs/` for the evolutionary trace.

The next cell shows a bounded readable preview. The complete, lossless prompt and
serialized program remain at the paths printed above.

In [3]:
print(learned_program_preview(OPTIMIZER))
print()
print("Frozen program/prompt consistency:", verify_prompt_artifact(OPTIMIZER))

Predictor: detect.predict
Learned instruction (9745 characters):
You are a provenance classifier. Your task is to determine whether a passage was most likely generated or rewritten by AI.

You will receive exactly one input field:

### text
The passage to classify.

Return exactly these two fields and no other content:

### reasoning
A concise explanation citing concrete, passage-specific evidence from diction, phrasing, rhythm, specificity, organization, transitions, or natural irregularities.

### is_ai
True or False

Use `True` when the passage was most likely AI-generated or AI-rewritten. Use `False` when it was most likely human-authored.

Evaluate the passage holistically. Do not decide from a single word, phrase, technical term, or superficial feature. Grammatical polish, formality, repetition, promotional language, parenthetical definitions, instructional structure, and technical sophistication are not sufficient by themselves. Conversely, concrete technical details do not auto

## Run it yourself (explicit opt-in)

The default `inspect` mode is offline and free. For a live run, first install from
the repository root with `python -m pip install -r requirements.txt`, configure
`OPENAI_API_KEY`, and restart the kernel.

- Bounded code-path check: launch Jupyter with `CHAPTER06_RUN=smoke`.
- Complete frozen-split rerun: launch Jupyter with `CHAPTER06_RUN=full`.

A full run is intentionally not triggered by opening or choosing “Run All”: it can
take minutes or incur model charges. The weight optimizers download and train a
small Qwen model locally through MPS/CPU and require the optional training stack. Live
artifacts are written to `chapter06/results/runs/<profile>/<optimizer>/<run-id>/`.
Rebuild the comparison afterward with `python -m chapter06.summarize_optimizer_results`.

In [4]:
if RUN_MODE == "inspect":
    print("Live run skipped. Set CHAPTER06_RUN=smoke or CHAPTER06_RUN=full explicitly.")
elif RUN_MODE in {"smoke", "full"}:
    from chapter06.optimizer_experiment import run_experiment

    live_result = run_experiment(OPTIMIZER, profile_name=RUN_MODE)
    print(live_result)
else:
    raise ValueError("CHAPTER06_RUN must be inspect, smoke, or full")

Live run skipped. Set CHAPTER06_RUN=smoke or CHAPTER06_RUN=full explicitly.
